# BARC GeoPackage → DataFrame

Loads the BARC GeoPackage into DataFrames using `barc_fields` / `barc_labels` from `preprocessing/loader.py`.

- `fields`: field polygons + metadata (label columns removed, GeoDataFrame)
- `labels`: validation labels (field_id, year, harvest_type). **Do not use them for training or feature engineering.**

In [ ]:
import sys
from pathlib import Path

# Add the repo root to the import path so the notebook works from any location
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "preprocessing" / "loader.py").exists())
sys.path.insert(0, str(ROOT))

import pandas as pd
from preprocessing.loader import barc_fields, barc_labels

GPKG = ROOT / "04_Evaluation" / "BARC_Corn.2021-2023.gpkg"
GPKG.exists()

## 1. Field polygons (no labels)

In [ ]:
fields = barc_fields(GPKG)   # GeoDataFrame
print(fields.shape, fields.crs.name)
fields.head()

A plain DataFrame without geometry, with area and centroid (lat/lon) added, for analysis that does not need polygons.

In [ ]:
utm = fields.to_crs(fields.estimate_utm_crs())
centroid_ll = utm.geometry.centroid.to_crs(4326)

fields_df = (pd.DataFrame(fields.drop(columns="geometry"))
             .assign(area_ha=utm.geometry.area / 1e4,
                     lon=centroid_ll.x.values,
                     lat=centroid_ll.y.values))
fields_df.head()

## 2. Labels (validation only)

In [ ]:
labels = barc_labels(GPKG)
print(labels.shape)
labels.groupby(["year", "harvest_type"]).size().unstack()

## 3. Field × year join (for validation and EDA)

In [ ]:
df = labels.merge(fields_df, on="field_id", how="left")
df.head()